In [ ]:
# ============================================================
# MEMBER B — BLIP-2 CAPTION GENERATION
# T4 x2 GPU | Resume-safe | Multi-checkpoint
# ============================================================

# =========================
# 1. INSTALL
# =========================
!pip install -q transformers accelerate Pillow pandas tqdm

# =========================
# 2. IMPORTS
# =========================
import os
import gc
import csv
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# =========================
# 3. PATHS
# =========================
CSV_PATH  = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs/blip_input_annotations.csv"
CROP_ROOT = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs/cropped_products"

OUTPUT_DIR   = "/kaggle/working/memberB_outputs"
CAPTIONS_CSV = os.path.join(OUTPUT_DIR, "captions.csv")
PROGRESS_LOG = os.path.join(OUTPUT_DIR, "progress_log.txt")
ERROR_LOG    = os.path.join(OUTPUT_DIR, "error_log.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# path remapping: CSV was written on /kaggle/working, files live in /kaggle/input
OLD_ROOT = "/kaggle/working/memberA_outputs"
NEW_ROOT = "/kaggle/input/datasets/nipunv23/croppedimages/memberA_outputs"

def remap_path(p):
    if pd.isna(p) or str(p).strip() == "":
        return None
    return str(p).replace(OLD_ROOT, NEW_ROOT)

# =========================
# 4. GPU SETUP
# =========================
print("\n--- GPU Info ---")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MULTI_GPU = torch.cuda.device_count() > 1
print(f"\nUsing multi-GPU: {USE_MULTI_GPU}")

# =========================
# 5. LOAD CSV + BUILD JOBS
# =========================
print("\nLoading annotation CSV...")
df = pd.read_csv(CSV_PATH)
print(f"Total rows in CSV: {len(df)}")

crop_cols = {
    "upper_body": "upper_body_crop",
    "lower_body": "lower_body_crop",
    "full_body":  "full_body_crop",
}

jobs = []
missing_on_disk = 0

for crop_class, col in crop_cols.items():
    subset = df[["original_image", col]].copy()
    subset = subset.dropna(subset=[col])
    subset = subset[subset[col].str.strip() != ""]
    for _, row in subset.iterrows():
        fixed_path = remap_path(row[col])
        if fixed_path is None:
            continue
        if os.path.exists(fixed_path):
            jobs.append({
                "original_image": row["original_image"],
                "image_name":     os.path.basename(fixed_path),
                "image_path":     fixed_path,
                "crop_class":     crop_class,
            })
        else:
            missing_on_disk += 1

print(f"Valid jobs found  : {len(jobs)}")
print(f"Missing on disk   : {missing_on_disk}")

# =========================
# 6. RESUME LOGIC
# ============================================================
# Check captions.csv for already-done image_names
# so a crashed session resumes without redoing work
# ============================================================
done_set = set()

if os.path.exists(CAPTIONS_CSV):
    try:
        done_df = pd.read_csv(CAPTIONS_CSV)
        done_set = set(done_df["image_name"].tolist())
        print(f"\nResume mode: {len(done_set)} captions already saved — skipping these.")
    except Exception as e:
        print(f"Could not read existing captions.csv ({e}) — starting fresh.")

remaining_jobs = [j for j in jobs if j["image_name"] not in done_set]
print(f"Jobs remaining    : {len(remaining_jobs)}")

if len(remaining_jobs) == 0:
    print("All jobs already done! Check captions.csv")
    raise SystemExit

# =========================
# 7. LOAD BLIP-2
# ============================================================
# blip2-opt-2.7b: lighter model, fits T4x2 in fp16 easily
# device_map="auto" with multi-GPU spreads model layers
# across both GPUs automatically
# ============================================================
print("\nLoading BLIP-2 processor...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

print("Loading BLIP-2 model (fp16)...")

if USE_MULTI_GPU:
    # device_map="auto" splits model layers across both T4s
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16,
    ).to(DEVICE)

model.eval()
print("BLIP-2 ready.")

# with device_map="auto", inputs go to the first device
INPUT_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# =========================
# 8. STORAGE SAFETY SETUP
# ============================================================
# Three layers of storage safety:
#
# Layer 1 — Line-buffered open() write after every caption
# Layer 2 — flush() + fsync() every FLUSH_EVERY batches
# Layer 3 — Full DataFrame checkpoint every CHECKPOINT_EVERY batches
#
# This means even if Kaggle kills the session mid-run,
# you lose at most FLUSH_EVERY * BATCH_SIZE captions
# ============================================================

BATCH_SIZE       = 16   # T4x2 with fp16 can handle 16 safely
FLUSH_EVERY      = 50   # flush to disk every 50 batches (~800 images)
CHECKPOINT_EVERY = 200  # full df save every 200 batches (~3200 images)

# Write header only if starting fresh
write_header = not os.path.exists(CAPTIONS_CSV) or len(done_set) == 0

out_f = open(CAPTIONS_CSV, "a", newline="", buffering=1)
writer = csv.writer(out_f, quoting=csv.QUOTE_ALL)

if write_header:
    writer.writerow(["original_image", "image_name", "crop_class", "caption"])
    out_f.flush()
    os.fsync(out_f.fileno())
    print(f"Created fresh captions.csv with header.")
else:
    print(f"Appending to existing captions.csv.")

# progress + error logs
prog_f  = open(PROGRESS_LOG, "a", buffering=1)
error_f = open(ERROR_LOG,    "a", buffering=1)

def log_progress(msg):
    print(msg)
    prog_f.write(msg + "\n")
    prog_f.flush()

def log_error(msg):
    print(f"[ERROR] {msg}")
    error_f.write(msg + "\n")
    error_f.flush()

# =========================
# 9. MAIN CAPTION LOOP
# =========================
log_progress(f"\nStarting caption generation — {len(remaining_jobs)} jobs")

total_done   = 0
total_errors = 0
batch_count  = 0

for i in tqdm(
    range(0, len(remaining_jobs), BATCH_SIZE),
    desc="Captioning"
):
    batch      = remaining_jobs[i : i + BATCH_SIZE]
    images     = []
    valid_jobs = []

    # --- load images for this batch ---
    for job in batch:
        try:
            img = Image.open(job["image_path"]).convert("RGB")
            images.append(img)
            valid_jobs.append(job)
        except Exception as e:
            log_error(f"Image load failed: {job['image_path']} | {e}")
            total_errors += 1
            continue

    if not images:
        continue

    # --- run BLIP-2 ---
    try:
        inputs = processor(
            images=images,
            return_tensors="pt"
        ).to(INPUT_DEVICE, torch.float16)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=40,
                num_beams=4,
                length_penalty=1.0,
                repetition_penalty=1.2,
            )

        captions = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # --- LAYER 1: write each row immediately ---
        for job, caption in zip(valid_jobs, captions):
            writer.writerow([
                job["original_image"],
                job["image_name"],
                job["crop_class"],
                caption.strip()
            ])
            total_done += 1

    except Exception as e:
        log_error(f"Batch {batch_count} inference failed: {e}")
        total_errors += 1
        # free memory and continue — don't crash entire run
        torch.cuda.empty_cache()
        gc.collect()
        continue

    batch_count += 1

    # --- LAYER 2: flush + fsync every FLUSH_EVERY batches ---
    if batch_count % FLUSH_EVERY == 0:
        out_f.flush()
        os.fsync(out_f.fileno())
        log_progress(
            f"[Flush] Batch {batch_count} | "
            f"Done: {total_done} | Errors: {total_errors}"
        )

    # --- LAYER 3: full DataFrame checkpoint ---
    if batch_count % CHECKPOINT_EVERY == 0:
        out_f.flush()
        os.fsync(out_f.fileno())
        try:
            ckpt_df = pd.read_csv(CAPTIONS_CSV)
            ckpt_path = os.path.join(
                OUTPUT_DIR,
                f"captions_checkpoint_{total_done}.csv"
            )
            ckpt_df.to_csv(ckpt_path, index=False)
            log_progress(
                f"[Checkpoint] Saved {ckpt_path} "
                f"({len(ckpt_df)} rows)"
            )
        except Exception as e:
            log_error(f"Checkpoint save failed: {e}")

# =========================
# 10. FINAL FLUSH + CLOSE
# =========================
out_f.flush()
os.fsync(out_f.fileno())
out_f.close()
prog_f.close()
error_f.close()

log_progress(f"\nAll done. Total captioned: {total_done} | Errors: {total_errors}")

# =========================
# 11. FINAL VERIFICATION
# ============================================================
# Confirms exactly what got saved to disk
# Run this as a sanity check before handing off to Member C
# ============================================================
print("\n" + "=" * 60)
print("FINAL OUTPUT VERIFICATION")
print("=" * 60)

final_df = pd.read_csv(CAPTIONS_CSV)

print(f"Total rows in captions.csv : {len(final_df)}")
print(f"Columns                    : {list(final_df.columns)}")
print(f"\nCrop class distribution:")
print(final_df["crop_class"].value_counts().to_string())

print(f"\nNull/empty captions: {final_df['caption'].isna().sum()}")

print(f"\nSample captions per class:")
for cls in ["upper_body", "lower_body", "full_body"]:
    subset = final_df[final_df["crop_class"] == cls].head(3)
    print(f"\n  [{cls}]")
    for _, r in subset.iterrows():
        print(f"    {r['image_name'][:50]}")
        print(f"    → {r['caption']}")

print(f"\nFile size: {os.path.getsize(CAPTIONS_CSV) / 1e6:.2f} MB")
print(f"Saved at : {CAPTIONS_CSV}")

print("\n--- Files in output dir ---")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size/1e6:.2f} MB)")